# Instagram two-agent pipeline (Agent1 unchanged) + Tavily evidence + improved Agent2

- Agent1: splits text into **Discriminative Text** and **Residual Text** (kept identical to your previous logic).
- Tavily: searches for **contact/company legitimacy + scam reports** for discriminative samples.
- Agent2: makes final label using **(a) agent1 output + (b) Tavily evidence**.

> **Label convention used here (based on your `instagram_final_results` errors):**
> - **1 = benign / normal profile**
> - **0 = suspicious / potential scam / unverifiable contact**

Set environment variables before running:
- `OPENAI_API_KEY`, `OPENAI_BASE_URL` (optional), `TAVILY_API_KEY`


In [ ]:
import os
import time
from typing import List, Dict, Any, Tuple

import pandas as pd
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "Input your key")
OPENAI_BASE_URL = os.getenv("OPENAI_API_BASE", os.getenv("OPENAI_BASE_URL", "Input your URL"))

if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY env var")

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)

def retry_with_backoff(func, *args, retries=10, delay=2, **kwargs):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(delay)
                delay *= 2
            else:
                raise last_err

def response_API(user_prompt: str, model: str = "gpt-4o-mini", temperature: float = 0.0) -> str:
    def api_call():
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": user_prompt}],
            temperature=temperature,
        )
        return resp.choices[0].message.content

    return retry_with_backoff(api_call, retries=8, delay=2)


AGENT1_PROMPT = r"""
You are a text splitter/classifier.
TASK
Split the provided input text into TWO buckets:
1) Discriminative Text
2) Residual Text
NON-NEGOTIABLE RULES (VERY IMPORTANT)
- Do NOT add any commentary, explanation, bullets, or extra headings.
- Do NOT paraphrase, summarize, translate, rewrite, or normalize.
- Do NOT delete anything: every line from the input MUST appear exactly once
  in either Discriminative Text or Residual Text.
- Preserve the original order of lines exactly as in the input.
- Keep emojis and special characters exactly as-is.
- If you are unsure, put it into Residual Text (but still do not delete it).
HARD DISCRIMINATIVE RULES (must go to Discriminative Text)
If a line contains ANY of the following, it MUST be Discriminative Text:
- '@' (email address or social handle, e.g., name@gmail.com, @username)
- '#' (hashtag/topic tag, e.g., #SEO, #London, #Trending, #Crypto)
- A URL or domain (http, https, www, .com, .co, .uk, .net, linktr.ee, beacons.ai, etc.)
- Contact / collaboration / business signals:
  "DM", "inquiries", "enquiries", "contact", "email", "collab", "collaborate",
  "booking", "bookings", "manager", "managed by", "agent", "PR", "press",
  "WhatsApp", "WeChat", "Line", "Telegram", "Kakao", "Snap", "Discord"
- Payment/account/channel identifiers:
  bank names, card brands, PayPal, Venmo, CashApp, Alipay, WeChat Pay, IBAN,
  sort code, account number, routing number, wallet address, crypto address
- Any explicit call-to-action to message/click/link/pay/subscribe.
WHAT GOES TO RESIDUAL
Everything else that does not match the hard discriminative rules.
OUTPUT FORMAT (MUST MATCH EXACTLY)
Discriminative Text:
<copy lines here exactly>
Residual Text:
<copy remaining lines here exactly>
Remember:
- First line must be exactly: Discriminative Text:
- Then later exactly: Residual Text:
- Output ONLY these two sections, nothing else.
""".strip()

def agent1_process(original_text) -> Dict[str, str]:
    if not isinstance(original_text, str):
        original_text = ""

    full_prompt = AGENT1_PROMPT + "\n\n" + original_text

    try:
        response = response_API(full_prompt)
    except Exception as e:
        print("Agent1 API failed:", e)
        response = ""

    if not response.strip():
        return {"Discriminative Text": "", "Residual Text": original_text or ""}

    disc_lines, resid_lines = [], []
    current_section = None

    for line in response.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("Discriminative Text"):
            current_section = "D"
            continue
        if line.startswith("Residual Text"):
            current_section = "R"
            continue
        if current_section == "D":
            disc_lines.append(line)
        elif current_section == "R":
            resid_lines.append(line)

    return {
        "Discriminative Text": "\n".join(disc_lines),
        "Residual Text": "\n".join(resid_lines),
    }

def run_agent1_and_save_csv(input_csv: str, output_csv: str, sleep_s: float = 0.5):
    df = pd.read_csv(input_csv)

    disc_col, resid_col = [], []
    for idx, row in df.iterrows():
        text = row.get("text", "")
        result = agent1_process(text)
        disc_col.append(result["Discriminative Text"])
        resid_col.append(result["Residual Text"])

        if idx % 10 == 0:
            print(f"Agent1 processed {idx} samples")
        time.sleep(sleep_s)

    df["Discriminative Text"] = disc_col
    df["Residual Text"] = resid_col
    df.to_csv(output_csv, index=False)
    print("Agent1 CSV saved to:", output_csv)


AGENT2_PROMPT = r"""
You are Agent2, a strict binary classifier.
Return ONLY one digit: 0 or 1.

You are given:
- text: full Instagram bio
- Discriminative Text: extracted high-signal cues (may be empty)
- Residual Text: remaining context (low priority)

GOAL:
Match the dataset labels. In this dataset, many bios that look like creators/influencers
are still label=0. Therefore, you must NOT output 1 based on weak “creator vibes”.

A) PRIORITY & NORMALIZATION
1) Discriminative Text > text > Residual Text.
2) If Discriminative Text is empty, do NOT assume positive. Treat as unknown.
3) Ignore corrupted emoji/encoding artifacts like "ðŸ", "ğŸ" etc.

B) CORE DECISION: SCORE + THRESHOLD
Compute a score using ONLY explicit cues in the text.
Default output is 0 unless the score reaches the threshold.

Start score = 0.

Add STRONG POSITIVE signals (+3 each):
+3  Mentions a recognized institution/platform/media network explicitly:
    Examples: "AM 640", "iHeartRadio", "HGTV", "NBC", "radio", "TV presenter"
+3  Clearly indicates an official organization/brand/service venue with operations:
    Examples: salon + booking schedule (W-2-F / appointments / books closed),
    business address + official brand handle, store/clinic/studio name + services.
+3  Clear official topic authority brand like "@SCIENCE" + inquiries email.

Add MODERATE POSITIVE signals (+2 each):
+2  Explicit "Business Inquiries:" / "Press:" / "Media:" + contact email
+2  Two or more consistent external handles pointing to a brand ecosystem
    (e.g., multiple linked brand accounts + role like Founder/Owner)

Add WEAK signals (+1 each) [NEVER sufficient alone]:
+1  Has an email
+1  Has "collab", "DM for collabs"
+1  Has "content creator", "influencer", "blogger"
+1  Has TikTok/YouTube mention
+1  Has generic profession label (photographer, stylist, coach) without external verification

Add STRONG NEGATIVE signals (-3 each):
-3  Pure diary / personal vibe only, no operational or institutional cues:
    examples: "visual diary", "my life", random quotes only
-3  Generic lifestyle list only (travel | fashion | beauty | mindset) with no verifiable anchor
-3  Text is mostly hashtags/emojis with no concrete role/entity

Add MODERATE NEGATIVE signals (-2 each):
-2  Overly generic “creator template” bios:
    "lifestyle | travel | fashion | mindset | growth" + DM/email only
-2  Only location + personal descriptors (mom/wife/dog lover) without institution/entity
-2  Discriminative Text contains only social handles or hashtags, no real-world anchor

C) OUTPUT THRESHOLD
If score >= 4  -> output 1
Else           -> output 0

IMPORTANT SAFETY GUARDRAILS:
- Never output 1 if the ONLY evidence is: email + "collab" + "lifestyle/blogger".
  (This pattern is frequently label=0 in this dataset.)
- If uncertain, output 0 (because false positives dominate your current errors).
- Return ONLY 0 or 1. No explanation.
""".strip()

def agent2_judge_from_row(row: pd.Series, sleep_s: float = 0.0) -> int:
    disc = row.get("Discriminative Text", "")
    resid = row.get("Residual Text", "")

    if not isinstance(disc, str) or not disc.strip():
        return 1

    user_input = f"Discriminative Text:\n{disc}\n\nResidual Text:\n{resid}"

    resp = response_API(AGENT2_PROMPT + "\n\n" + user_input)
    time.sleep(sleep_s)

    try:
        val = int(resp.strip())
        return 1 if val == 1 else 0
    except Exception:
        return 1

def run_agent2_and_save_csv(agent1_csv: str, output_csv: str, sleep_s: float = 0.5):
    df = pd.read_csv(agent1_csv)

    preds = []
    for idx, row in df.iterrows():
        pred = agent2_judge_from_row(row)
        preds.append(pred)
        if idx % 10 == 0:
            print(f"Agent2 processed {idx} samples")
        time.sleep(sleep_s)

    df["agent2_pred"] = preds
    df.to_csv(output_csv, index=False)
    print("Final CSV saved to:", output_csv)

if __name__ == "__main__":
    input_csv = "/content/instagram_samples500.csv"
    after_a1 = "/content/instagram_after_agent1.csv"
    final_out = "/content/instagram_final_results_no_tavily.csv"

    run_agent1_and_save_csv(input_csv=input_csv, output_csv=after_a1)
    run_agent2_and_save_csv(agent1_csv=after_a1, output_csv=final_out)

Agent1 processed 0 samples
Agent1 processed 10 samples
Agent1 processed 20 samples
Agent1 processed 30 samples
Agent1 processed 40 samples
Agent1 processed 50 samples
Agent1 processed 60 samples
Agent1 processed 70 samples
Agent1 processed 80 samples
Agent1 processed 90 samples
Agent1 processed 100 samples
Agent1 processed 110 samples
Agent1 processed 120 samples
Agent1 processed 130 samples
Agent1 processed 140 samples
Agent1 processed 150 samples
Agent1 processed 160 samples
Agent1 processed 170 samples
Agent1 processed 180 samples
Agent1 processed 190 samples
Agent1 processed 200 samples
Agent1 processed 210 samples
Agent1 processed 220 samples
Agent1 processed 230 samples
Agent1 processed 240 samples
Agent1 processed 250 samples
Agent1 processed 260 samples
Agent1 processed 270 samples
Agent1 processed 280 samples
Agent1 processed 290 samples
Agent1 processed 300 samples
Agent1 processed 310 samples
Agent1 processed 320 samples
Agent1 processed 330 samples
Agent1 processed 340 samp

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Refactored to use a relative path to ensure cross-platform compatibility
df_result = pd.read_csv("/content/instagram_final_results_no_tavily.csv")

y_true = df_result["label"]
y_pred = df_result["agent2_pred"]

# Retain only rows where both ground truth labels and predictions are valid (non-null)
mask = y_true.notna() & y_pred.notna()

y_true_clean = y_true[mask]
y_pred_clean = y_pred[mask]

# Calculate core binary classification metrics
acc = accuracy_score(y_true_clean, y_pred_clean)
p   = precision_score(y_true_clean, y_pred_clean, average="binary")
r   = recall_score(y_true_clean, y_pred_clean, average="binary")
f1  = f1_score(y_true_clean, y_pred_clean, average="binary")

#F1-macro
f1_macro = f1_score(y_true_clean, y_pred_clean, average="macro")

print("Used rows:", mask.sum(), "/", len(df_result))
print("Accuracy:", acc,
      "Precision:", p,
      "Recall:", r,
      "F1:", f1,
      "F1-macro:", f1_macro)



Used rows: 500 / 500
Accuracy: 0.584 Precision: 0.40425531914893614 Recall: 0.3149171270718232 F1: 0.35403726708074534 F1-macro: 0.5236263031568918


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
